# 05 · Location lookup for the app

**Input:** `SRC.india_post_pincode_directory` (read-only share)
**Output:** `OUT.app_locations`: one row per PIN code and per district, with a robust centre point

The app resolves "800001" or "Darbhanga" to a point with this table. City names already resolve from
`facilities_clean`, so this table is optional, but without it PIN codes and districts with no facilities can't be found.
Columns: `kind` (pin / district), `name`, `state`, `lat`, `lon`, `n_points`, `source`.

In [0]:
%run ./00_config

In [0]:
pin = spark.table(f"{SRC}.india_post_pincode_directory")
pin.printSchema()
display(pin.limit(5))

In [0]:
from pyspark.sql import functions as F

cols = {c.lower(): c for c in pin.columns}

def pick(*names):
    for n in names:
        if n in cols:
            return cols[n]
    return None

PIN_COL = pick("pincode", "pin_code", "pin", "postcode")
LAT_COL = pick("latitude", "lat")
LON_COL = pick("longitude", "long", "lon", "lng")
DIST_COL = pick("district", "districtname", "district_name")
STATE_COL = pick("statename", "state_name", "state")
print(dict(pin=PIN_COL, lat=LAT_COL, lon=LON_COL, district=DIST_COL, state=STATE_COL))
assert PIN_COL and LAT_COL and LON_COL, "Check the schema above and set the column names by hand"

# Post-office points; many rows have 'NA' or junk coordinates -> try_cast + India box
pts = (pin.select(
            F.regexp_replace(F.col(PIN_COL).cast("string"), r"\.0$", "").alias("pin"),
            F.expr(f"try_cast(`{LAT_COL}` as double)").alias("lat"),
            F.expr(f"try_cast(`{LON_COL}` as double)").alias("lon"),
            (F.initcap(F.trim(F.col(DIST_COL))) if DIST_COL else F.lit(None).cast("string")).alias("district"),
            (F.initcap(F.trim(F.col(STATE_COL))) if STATE_COL else F.lit(None).cast("string")).alias("state"))
         .where(F.col("pin").rlike(r"^[1-9][0-9]{5}$")
                & F.col("lat").between(6, 37.5) & F.col("lon").between(68, 97.5)))

def centres(df, keys, kind, name_col):
    return (df.groupBy(*keys)
              .agg(F.percentile_approx("lat", 0.5).alias("lat"),
                   F.percentile_approx("lon", 0.5).alias("lon"),
                   F.count("*").alias("n_points"),
                   F.first("state", ignorenulls=True).alias("state_any"))
              .select(F.lit(kind).alias("kind"), F.col(name_col).alias("name"),
                      (F.col("state") if "state" in keys else F.col("state_any")).alias("state"),
                      "lat", "lon", "n_points",
                      F.lit("median of India Post office locations").alias("source")))

pin_centres = centres(pts, ["pin"], "pin", "pin")
parts = [pin_centres]
if DIST_COL:
    parts.append(centres(pts.where(F.col("district").isNotNull()), ["district", "state"], "district", "district"))

locations = parts[0]
for p in parts[1:]:
    locations = locations.unionByName(p)

(locations.write.mode("overwrite").option("overwriteSchema", "true")
          .saveAsTable(f"{OUT}.app_locations"))
display(spark.table(f"{OUT}.app_locations").groupBy("kind").count())

### Give the app read access
The Databricks App runs as its own service principal. After creating the app, copy its service principal
name/ID from the app page, then run in a SQL cell (replace the placeholder):

```sql
GRANT USE CATALOG ON CATALOG workspace TO `<app-service-principal>`;
GRANT USE SCHEMA ON SCHEMA workspace.datathon TO `<app-service-principal>`;
GRANT SELECT ON TABLE workspace.datathon.facilities_clean TO `<app-service-principal>`;
GRANT SELECT ON TABLE workspace.datathon.facility_service_evidence TO `<app-service-principal>`;
GRANT SELECT ON TABLE workspace.datathon.app_locations TO `<app-service-principal>`;
```